In [2]:
import os
import shutil
import random

SOURCE_DIR = r"E:\IDP project\Pothole Dataset"
OUTPUT_DIR = r"E:\IDP project\Dataset"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

random.seed(42)

# Create train/val/test folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DIR, "Images", split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "Labels", split), exist_ok=True)

# Find all images
images = [
    f for f in os.listdir(SOURCE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Total images found:", len(images))

# Check that every image has a label
missing_labels = []

for image in images:
    label = os.path.splitext(image)[0] + ".txt"
    label_path = os.path.join(SOURCE_DIR, label)

    if not os.path.exists(label_path):
        missing_labels.append(image)

if missing_labels:
    print("Missing labels:")
    for image in missing_labels:
        print(image)
    raise Exception("Some images do not have labels.")

print("All images have corresponding labels.")

# Shuffle images
random.shuffle(images)

# Calculate split sizes
total = len(images)
train_count = int(total * TRAIN_RATIO)
val_count = int(total * VAL_RATIO)

train_images = images[:train_count]
val_images = images[train_count:train_count + val_count]
test_images = images[train_count + val_count:]

# Function to copy images and labels
def copy_dataset(file_list, split):

    for image in file_list:

        # Image
        image_source = os.path.join(SOURCE_DIR, image)
        image_destination = os.path.join(
            OUTPUT_DIR, "Images", split, image
        )

        # Label
        label = os.path.splitext(image)[0] + ".txt"
        label_source = os.path.join(SOURCE_DIR, label)
        label_destination = os.path.join(
            OUTPUT_DIR, "Labels", split, label
        )

        shutil.copy2(image_source, image_destination)
        shutil.copy2(label_source, label_destination)


# Perform splitting
copy_dataset(train_images, "train")
copy_dataset(val_images, "val")
copy_dataset(test_images, "test")

print("\n===================================")
print("DATASET SPLIT COMPLETED")
print("===================================")
print("Total      :", total)
print("Train      :", len(train_images))
print("Validation :", len(val_images))
print("Test       :", len(test_images))
print("===================================")

Total images found: 1243
All images have corresponding labels.

DATASET SPLIT COMPLETED
Total      : 1243
Train      : 870
Validation : 186
Test       : 187


In [1]:
import os

base_dir = r"E:\IDP project\Dataset"

splits = ["train", "val", "test"]

for split in splits:
    image_dir = os.path.join(base_dir, "Images", split)
    label_dir = os.path.join(base_dir, "Labels", split)

    images = [
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    labels = [
        f for f in os.listdir(label_dir)
        if f.lower().endswith(".txt")
    ]

    print(f"\n{split.upper()}")
    print("Images :", len(images))
    print("Labels :", len(labels))


TRAIN
Images : 870
Labels : 870

VAL
Images : 186
Labels : 186

TEST
Images : 187
Labels : 187


In [2]:
import os

base_dir = r"E:\IDP project\Dataset"

invalid_labels = []

for split in ["train", "val", "test"]:

    label_dir = os.path.join(base_dir, "Labels", split)

    for filename in os.listdir(label_dir):

        if not filename.endswith(".txt"):
            continue

        filepath = os.path.join(label_dir, filename)

        with open(filepath, "r") as f:
            lines = f.readlines()

        for line_number, line in enumerate(lines, start=1):

            values = line.strip().split()

            # YOLO format must have 5 values
            if len(values) != 5:
                invalid_labels.append(
                    (split, filename, line_number, "Wrong number of values")
                )
                continue

            try:
                class_id = int(values[0])
                x, y, width, height = map(float, values[1:])
            except:
                invalid_labels.append(
                    (split, filename, line_number, "Non-numeric value")
                )
                continue

            # Check class
            if class_id != 0:
                invalid_labels.append(
                    (split, filename, line_number, "Class is not 0")
                )

            # Check bounding box
            if width <= 0 or height <= 0:
                invalid_labels.append(
                    (split, filename, line_number, "Zero/negative width or height")
                )

            if not (0 <= x <= 1 and 0 <= y <= 1):
                invalid_labels.append(
                    (split, filename, line_number, "Center outside range")
                )

            if not (0 < width <= 1 and 0 < height <= 1):
                invalid_labels.append(
                    (split, filename, line_number, "Width/height outside range")
                )

print("Invalid annotations found:", len(invalid_labels))

for item in invalid_labels[:20]:
    print(item)

Invalid annotations found: 2
('train', 'img-415.txt', 3, 'Zero/negative width or height')
('train', 'img-415.txt', 3, 'Width/height outside range')


In [3]:
import os

dataset_path = r"E:\IDP project\Dataset"

yaml_content = f"""path: "{dataset_path.replace(chr(92), '/')}"
train: Images/train
val: Images/val
test: Images/test

nc: 1
names:
  0: pothole
"""

yaml_file = os.path.join(dataset_path, "data.yaml")

with open(yaml_file, "w") as f:
    f.write(yaml_content)

print("data.yaml created successfully!")
print("\nLocation:")
print(yaml_file)

print("\nContents:")
print(yaml_content)

data.yaml created successfully!

Location:
E:\IDP project\Dataset\data.yaml

Contents:
path: "E:/IDP project/Dataset"
train: Images/train
val: Images/val
test: Images/test

nc: 1
names:
  0: pothole



In [4]:
!pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable


In [5]:
from ultralytics import YOLO

print("Ultralytics imported successfully!")

model = YOLO("yolov8n.pt")

print("YOLOv8 Nano model loaded successfully!")

Ultralytics imported successfully!
YOLOv8 Nano model loaded successfully!


In [6]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano
model = YOLO("yolov8n.pt")

# Train the model
results = model.train(
    data=r"E:\IDP project\Dataset\data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="pothole_yolov8_baseline",
    patience=10
)

Ultralytics 8.4.143  Python-3.13.9 torch-2.14.0+cpu CPU (AMD Ryzen 5 8645HS w/ Radeon 760M Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\IDP project\Dataset\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pothole_yolov8_baseline, nb

ValueError: [34m[1mtrain: [0mNo labels found in E:\IDP project\Dataset\Images\train.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.

In [7]:
import os

dataset_path = r"E:\IDP project\Dataset"

yaml_content = """path: "E:/IDP project/Dataset"

train: Images/train
val: Images/val
test: Images/test

nc: 1
names:
  0: pothole
"""

yaml_file = os.path.join(dataset_path, "data.yaml")

with open(yaml_file, "w") as f:
    f.write(yaml_content)

print("data.yaml updated!")
print(yaml_content)

data.yaml updated!
path: "E:/IDP project/Dataset"

train: Images/train
val: Images/val
test: Images/test

nc: 1
names:
  0: pothole



In [10]:
import os

dataset = r"E:\IDP project\Dataset"

print("Contents of Dataset:")
print(os.listdir(dataset))

print("\nImages folder:")
print(os.listdir(os.path.join(dataset, "images")))

print("\nLabels folder:")
print(os.listdir(os.path.join(dataset, "labels")))

Contents of Dataset:
['data.yaml', 'Images', 'Labels']

Images folder:
['test', 'train', 'val']

Labels folder:
['test', 'train', 'val']


In [15]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=r"E:\IDP project\Dataset\data.yaml",
    epochs=1,
    imgsz=640,
    batch=16,
    name="pothole_test",
    patience=1
)

Ultralytics 8.4.143  Python-3.13.9 torch-2.14.0+cpu CPU (AMD Ryzen 5 8645HS w/ Radeon 760M Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\IDP project\Dataset\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pothole_test, nbs=64, nms=No

In [16]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano
model = YOLO("yolov8n.pt")

# Full training
results = model.train(
    data=r"E:\IDP project\Dataset\data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    workers=4,
    name="pothole_yolov8_baseline",
    pretrained=True,
    plots=True
)

Ultralytics 8.4.143  Python-3.13.9 torch-2.14.0+cpu CPU (AMD Ryzen 5 8645HS w/ Radeon 760M Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\IDP project\Dataset\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pothole_yolov8_baseline-2, 

In [17]:
from ultralytics import YOLO

# Load the best trained model
model = YOLO(
    r"C:\Users\aravi\Road Pothole\runs\detect\pothole_yolov8_baseline-2\weights\best.pt"
)

print("Best YOLOv8n model loaded successfully!")

Best YOLOv8n model loaded successfully!


In [18]:
test_results = model.val(
    data=r"E:\IDP project\Dataset\data.yaml",
    split="test",
    imgsz=640,
    batch=16
)

print("\nTEST RESULTS")
print("-------------------------")
print("Precision :", test_results.box.mp)
print("Recall    :", test_results.box.mr)
print("mAP@50    :", test_results.box.map50)
print("mAP@50-95 :", test_results.box.map)

Ultralytics 8.4.143  Python-3.13.9 torch-2.14.0+cpu CPU (AMD Ryzen 5 8645HS w/ Radeon 760M Graphics)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.20.2 ms, read: 5.85.0 MB/s, size: 69.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning E:\IDP project\Dataset\labels\test... 187 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 187/187 227.7it/s 0.8s0.1s
val: New cache created: E:\IDP project\Dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.1it/s 10.7s.0s
                   all        187        589      0.765      0.526      0.644      0.326
Speed: 1.3ms preprocess, 42.9ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\aravi\Road Pothole\runs\detect\val

TEST RESULTS
------------